In [1]:
import os
import random
import numpy as np
import pandas as pd

from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.model_selection import GroupShuffleSplit

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

PyTorch: 2.13.0+cpu
CUDA available: False
Device: cpu


In [2]:
BASE_PATH = "/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026"

SENSOR_HZ = 50
VIDEO_FPS = 30

SENSOR_WINDOW = 50
VIDEO_WINDOW = 15

WINDOW_STRIDE = 25

NUM_CLASSES = 19

D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 3

DROPOUT = 0.2

BATCH_SIZE = 128
EPOCHS = 30

LR = 2e-4
WEIGHT_DECAY = 1e-4

SEED = 42

print("Configuration loaded.")

Configuration loaded.


In [3]:
def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [4]:
LABEL_TO_ID = {
    "null": 0,
    "jogging": 1,
    "jogging (rotating arms)": 2,
    "jogging (skipping)": 3,
    "jogging (sidesteps)": 4,
    "jogging (butt-kicks)": 5,
    "stretching (triceps)": 6,
    "stretching (lunging)": 7,
    "stretching (shoulders)": 8,
    "stretching (hamstrings)": 9,
    "stretching (lumbar rotation)": 10,
    "push-ups": 11,
    "push-ups (complex)": 12,
    "sit-ups": 13,
    "sit-ups (complex)": 14,
    "burpees": 15,
    "lunges": 16,
    "lunges (complex)": 17,
    "bench-dips": 18
}

ID_TO_LABEL = {
    v: k
    for k, v in LABEL_TO_ID.items()
}

assert len(LABEL_TO_ID) == 19

print(ID_TO_LABEL)

{0: 'null', 1: 'jogging', 2: 'jogging (rotating arms)', 3: 'jogging (skipping)', 4: 'jogging (sidesteps)', 5: 'jogging (butt-kicks)', 6: 'stretching (triceps)', 7: 'stretching (lunging)', 8: 'stretching (shoulders)', 9: 'stretching (hamstrings)', 10: 'stretching (lumbar rotation)', 11: 'push-ups', 12: 'push-ups (complex)', 13: 'sit-ups', 14: 'sit-ups (complex)', 15: 'burpees', 16: 'lunges', 17: 'lunges (complex)', 18: 'bench-dips'}


In [5]:
SENSOR_MAPPINGS = {

    "right_arm": [
        "right_arm_acc_x",
        "right_arm_acc_y",
        "right_arm_acc_z"
    ],

    "right_leg": [
        "right_leg_acc_x",
        "right_leg_acc_y",
        "right_leg_acc_z"
    ],

    "left_leg": [
        "left_leg_acc_x",
        "left_leg_acc_y",
        "left_leg_acc_z"
    ],

    "left_arm": [
        "left_arm_acc_x",
        "left_arm_acc_y",
        "left_arm_acc_z"
    ]
}

LOCATION_TO_ID = {
    "right_arm": 0,
    "right_leg": 1,
    "left_leg": 2,
    "left_arm": 3
}

In [6]:
train_sensor_dir = os.path.join(
    BASE_PATH,
    "train",
    "inertial_feat"
)

train_video_dir = os.path.join(
    BASE_PATH,
    "train",
    "videomae_feat"
)

recordings = []

for filename in sorted(os.listdir(train_sensor_dir)):

    if not filename.endswith(".csv"):
        continue

    stem = filename[:-4]

    video_file = stem + ".npy"

    sensor_path = os.path.join(
        train_sensor_dir,
        filename
    )

    video_path = os.path.join(
        train_video_dir,
        video_file
    )

    if not os.path.exists(video_path):
        print("Missing video:", stem)
        continue

    recordings.append({
        "recording": stem,
        "sensor_path": sensor_path,
        "video_path": video_path
    })

print("Total recordings:", len(recordings))

for r in recordings:
    print(r["recording"])

Total recordings: 24
sbj_0
sbj_0_2
sbj_1
sbj_10
sbj_11
sbj_12
sbj_13
sbj_14
sbj_14_2
sbj_15
sbj_16
sbj_17
sbj_18
sbj_19
sbj_2
sbj_20
sbj_21
sbj_3
sbj_4
sbj_5
sbj_6
sbj_7
sbj_8
sbj_9


In [7]:
def build_recording_index(recording):

    df = pd.read_csv(
        recording["sensor_path"],
        usecols=[
            "sbj_id",
            "label"
        ]
    )

    labels = df["label"].fillna("null").to_numpy()

    records = []

    n = len(df)

    for start in range(
        0,
        n - SENSOR_WINDOW + 1,
        WINDOW_STRIDE
    ):

        window_labels = labels[
            start:start + SENSOR_WINDOW
        ]

        counts = Counter(window_labels)

        majority_label, majority_count = (
            counts.most_common(1)[0]
        )

        # Require at least 80% agreement.
        if majority_count < 40:
            continue

        label_id = LABEL_TO_ID[majority_label]

        sbj_id = int(
            df["sbj_id"].iloc[0]
        )

        for location in SENSOR_MAPPINGS:

            records.append({
                "recording": recording["recording"],
                "sensor_path": recording["sensor_path"],
                "video_path": recording["video_path"],
                "sbj_id": sbj_id,
                "start": start,
                "sensor_location": location,
                "location_id": LOCATION_TO_ID[location],
                "label": majority_label,
                "label_id": label_id
            })

    return records

In [8]:
all_records = []

for i, recording in enumerate(recordings):

    rec_records = build_recording_index(
        recording
    )

    all_records.extend(rec_records)

    print(
        f"{i+1:02d}/{len(recordings)} "
        f"{recording['recording']}: "
        f"{len(rec_records)} samples"
    )

train_index = pd.DataFrame(all_records)

print("\nTotal training samples:", len(train_index))

print("\nClass distribution:")

print(
    train_index["label"]
    .value_counts()
)

01/24 sbj_0: 22032 samples
02/24 sbj_0_2: 17748 samples
03/24 sbj_1: 21888 samples


/tmp/ipykernel_58/2948890058.py:3: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


04/24 sbj_10: 31620 samples
05/24 sbj_11: 17592 samples
06/24 sbj_12: 25872 samples
07/24 sbj_13: 31476 samples
08/24 sbj_14: 25316 samples
09/24 sbj_14_2: 26568 samples
10/24 sbj_15: 20616 samples
11/24 sbj_16: 27852 samples
12/24 sbj_17: 24412 samples
13/24 sbj_18: 20812 samples
14/24 sbj_19: 16688 samples
15/24 sbj_2: 28092 samples
16/24 sbj_20: 15092 samples
17/24 sbj_21: 20212 samples
18/24 sbj_3: 32820 samples
19/24 sbj_4: 25208 samples
20/24 sbj_5: 16860 samples
21/24 sbj_6: 19784 samples
22/24 sbj_7: 22560 samples
23/24 sbj_8: 19748 samples
24/24 sbj_9: 17124 samples

Total training samples: 547992

Class distribution:
label
null                            217028
jogging                          19880
jogging (sidesteps)              19584
stretching (lunging)             19388
lunges (complex)                 19292
sit-ups (complex)                19124
sit-ups                          18920
lunges                           18768
stretching (triceps)             18508
stretchi

In [9]:
unique_subjects = sorted(
    train_index["sbj_id"].unique()
)

print(
    "Training subjects:",
    unique_subjects
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        train_index,
        groups=train_index["sbj_id"]
    )
)

train_df = train_index.iloc[
    train_idx
].reset_index(drop=True)

val_df = train_index.iloc[
    val_idx
].reset_index(drop=True)

print("\nTrain subjects:")
print(sorted(train_df["sbj_id"].unique()))

print("\nValidation subjects:")
print(sorted(val_df["sbj_id"].unique()))

print("\nTrain samples:", len(train_df))
print("Validation samples:", len(val_df))

Training subjects: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21)]

Train subjects:
[np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21)]

Validation subjects:
[np.int64(0), np.int64(1), np.int64(8), np.int64(13), np.int64(15)]

Train samples: 414484
Validation samples: 133508


In [10]:
sensor_columns = [
    col
    for cols in SENSOR_MAPPINGS.values()
    for col in cols
]

# Remove duplicates while preserving order
sensor_columns = list(
    dict.fromkeys(sensor_columns)
)

train_sensor_values = []

for path in train_df["sensor_path"].unique():

    temp = pd.read_csv(
        path,
        usecols=sensor_columns
    )

    train_sensor_values.append(
        temp.values.astype(np.float32)
    )

train_sensor_values = np.concatenate(
    train_sensor_values,
    axis=0
)

sensor_mean = train_sensor_values.mean(
    axis=0
)

sensor_std = train_sensor_values.std(
    axis=0
)

sensor_std[
    sensor_std < 1e-6
] = 1.0

print("Sensor mean:")
print(sensor_mean)

print("\nSensor std:")
print(sensor_std)

Sensor mean:
[ 0.41761622  0.41242546 -0.23019284  0.7253878  -0.00226478  0.15965568
  0.62616706  0.03128885  0.15277538         nan         nan         nan]

Sensor std:
[0.7663313  0.6375716  0.49619004 0.6879926  0.7447274  0.44381988
 0.7689527  0.74331576 0.44784287        nan        nan        nan]


In [11]:
class WearDataset(Dataset):

    def __init__(
        self,
        index_df,
        sensor_mean,
        sensor_std
    ):

        self.df = index_df.reset_index(
            drop=True
        )

        self.sensor_mean = sensor_mean
        self.sensor_std = sensor_std

        self._sensor_cache = {}
        self._video_cache = {}

    def __len__(self):

        return len(self.df)

    def _load_recording(self, recording):

        row = self.df.iloc[recording]

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        sensor_path = row["sensor_path"]
        video_path = row["video_path"]

        # --------------------------------
        # Load sensor CSV
        # --------------------------------

        if sensor_path not in self._sensor_cache:

            sensor_df = pd.read_csv(
                sensor_path
            )

            self._sensor_cache[sensor_path] = (
                sensor_df
            )

        sensor_df = self._sensor_cache[
            sensor_path
        ]

        # --------------------------------
        # Load video with memory mapping
        # --------------------------------

        if video_path not in self._video_cache:

            self._video_cache[video_path] = (
                np.load(
                    video_path,
                    mmap_mode="r"
                )
            )

        video = self._video_cache[
            video_path
        ]

        start = int(row["start"])

        # --------------------------------
        # Sensor
        # --------------------------------

        columns = SENSOR_MAPPINGS[
            row["sensor_location"]
        ]

        sensor = sensor_df.loc[
            start:start + SENSOR_WINDOW - 1,
            columns
        ].values.astype(np.float32)

        # --------------------------------
        # Normalize
        # --------------------------------

        column_indices = [
            sensor_columns.index(c)
            for c in columns
        ]

        mean = self.sensor_mean[
            column_indices
        ]

        std = self.sensor_std[
            column_indices
        ]

        sensor = (
            sensor - mean
        ) / std

        # --------------------------------
        # Video alignment
        # --------------------------------

        video_start = int(
            start * VIDEO_FPS / SENSOR_HZ
        )

        # Exactly 30 frames correspond
        # to a 1-second sensor window.
        #
        # Use the central 15 feature vectors
        # so VideoMAE's temporal context stays
        # inside the 1-second interval.

        video_start += 8
        video_end = video_start + VIDEO_WINDOW

        video_window = np.asarray(
            video[
                video_start:video_end
            ],
            dtype=np.float32
        )

        # --------------------------------
        # Shape checks
        # --------------------------------

        assert sensor.shape == (50, 3)

        assert video_window.shape == (
            15,
            768
        )

        # --------------------------------
        # Return
        # --------------------------------

        return {
            "sensor": torch.tensor(
                sensor,
                dtype=torch.float32
            ),

            "video": torch.tensor(
                video_window,
                dtype=torch.float32
            ),

            "location": torch.tensor(
                int(row["location_id"]),
                dtype=torch.long
            ),

            "label": torch.tensor(
                int(row["label_id"]),
                dtype=torch.long
            )
        }

In [12]:
train_dataset = WearDataset(
    train_df,
    sensor_mean,
    sensor_std
)

val_dataset = WearDataset(
    val_df,
    sensor_mean,
    sensor_std
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 414484
Validation: 133508


In [13]:
sample = train_dataset[0]

print("Sensor:", sample["sensor"].shape)
print("Video:", sample["video"].shape)
print("Location:", sample["location"])
print("Label:", sample["label"])

Sensor: torch.Size([50, 3])
Video: torch.Size([15, 768])
Location: tensor(0)
Label: tensor(0)


/tmp/ipykernel_58/1718166332.py:41: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  sensor_df = pd.read_csv(


In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

batch = next(iter(train_loader))

print(
    "Sensor batch:",
    batch["sensor"].shape
)

print(
    "Video batch:",
    batch["video"].shape
)

print(
    "Labels:",
    batch["label"].shape
)

Sensor batch: torch.Size([128, 50, 3])
Video batch: torch.Size([128, 15, 768])
Labels: torch.Size([128])


In [15]:
class MultimodalTransformer(nn.Module):

    def __init__(
        self,
        num_classes=19,
        d_model=128,
        nhead=4,
        num_layers=3,
        dropout=0.2
    ):

        super().__init__()

        # --------------------------------
        # Modality projections
        # --------------------------------

        self.sensor_projection = nn.Sequential(
            nn.Linear(3, d_model),
            nn.LayerNorm(d_model)
        )

        self.video_projection = nn.Sequential(
            nn.Linear(768, d_model),
            nn.LayerNorm(d_model)
        )

        # --------------------------------
        # Location embedding
        # --------------------------------

        self.location_embedding = nn.Embedding(
            4,
            d_model
        )

        # --------------------------------
        # CLS token
        # --------------------------------

        self.cls_token = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

        # --------------------------------
        # Positional embeddings
        #
        # 1 CLS + 50 sensor + 15 video
        # --------------------------------

        self.position_embedding = nn.Parameter(
            torch.randn(
                1,
                66,
                d_model
            ) * 0.02
        )

        # --------------------------------
        # Modality embeddings
        #
        # 0 = CLS
        # 1 = sensor
        # 2 = video
        # --------------------------------

        self.modality_embedding = nn.Embedding(
            3,
            d_model
        )

        # --------------------------------
        # Transformer
        # --------------------------------

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.final_norm = nn.LayerNorm(
            d_model
        )

        # --------------------------------
        # Classification head
        # --------------------------------

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(
        self,
        sensor,
        video,
        location
    ):

        B = sensor.size(0)

        # Sensor tokens
        sensor_tokens = self.sensor_projection(
            sensor
        )

        # Video tokens
        video_tokens = self.video_projection(
            video
        )

        # Location information
        location_embedding = self.location_embedding(
            location
        ).unsqueeze(1)

        sensor_tokens = (
            sensor_tokens
            + location_embedding
        )

        # CLS
        cls = self.cls_token.expand(
            B, -1, -1
        )

        # Concatenate
        x = torch.cat(
            [
                cls,
                sensor_tokens,
                video_tokens
            ],
            dim=1
        )

        # Modality IDs
        cls_mod = torch.zeros(
            B, 1,
            dtype=torch.long,
            device=x.device
        )

        sensor_mod = torch.ones(
            B,
            50,
            dtype=torch.long,
            device=x.device
        )

        video_mod = torch.full(
            (B, 15),
            2,
            dtype=torch.long,
            device=x.device
        )

        modality_ids = torch.cat(
            [
                cls_mod,
                sensor_mod,
                video_mod
            ],
            dim=1
        )

        x = (
            x
            + self.position_embedding[:, :x.size(1)]
            + self.modality_embedding(
                modality_ids
            )
        )

        # Transformer
        x = self.transformer(x)

        x = self.final_norm(x)

        # CLS representation
        cls_output = x[:, 0]

        logits = self.classifier(
            cls_output
        )

        return logits

In [16]:
model = MultimodalTransformer(
    num_classes=NUM_CLASSES,
    d_model=D_MODEL,
    nhead=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

print(model)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"\nTotal parameters: "
    f"{total_params:,}"
)

MultimodalTransformer(
  (sensor_projection): Sequential(
    (0): Linear(in_features=3, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (video_projection): Sequential(
    (0): Linear(in_features=768, out_features=128, bias=True)
    (1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (location_embedding): Embedding(4, 128)
  (modality_embedding): Embedding(3, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, el

/tmp/ipykernel_58/2487407591.py:86: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [17]:
train_counts = (
    train_df["label_id"]
    .value_counts()
    .reindex(
        range(NUM_CLASSES),
        fill_value=1
    )
    .values
)

class_weights = (
    len(train_df)
    /
    (
        NUM_CLASSES
        * train_counts
    )
)

# Don't allow extremely large weights
class_weights = np.clip(
    class_weights,
    0.25,
    5.0
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(DEVICE)

print(class_weights)

tensor([0.2500, 1.5247, 1.7480, 1.6294, 1.4901, 1.7469, 1.5891, 1.5049, 1.6338,
        1.5753, 1.5813, 1.7374, 1.6183, 1.5961, 1.5183, 1.5524, 1.6246, 1.5690,
        1.7741], device='cuda:0')


In [18]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

In [19]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0
    all_preds = []
    all_targets = []

    for batch in loader:

        sensor = batch["sensor"].to(
            DEVICE,
            non_blocking=True
        )

        video = batch["video"].to(
            DEVICE,
            non_blocking=True
        )

        location = batch["location"].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch["label"].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            sensor,
            video,
            location
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += (
            loss.item()
            * targets.size(0)
        )

        preds = logits.argmax(
            dim=1
        )

        all_preds.extend(
            preds.detach().cpu().numpy()
        )

        all_targets.extend(
            targets.detach().cpu().numpy()
        )

    avg_loss = (
        total_loss / len(loader.dataset)
    )

    macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        all_targets,
        all_preds
    )

    return (
        avg_loss,
        macro_f1,
        accuracy
    )

In [20]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0

    all_preds = []
    all_targets = []

    for batch in loader:

        sensor = batch["sensor"].to(
            DEVICE,
            non_blocking=True
        )

        video = batch["video"].to(
            DEVICE,
            non_blocking=True
        )

        location = batch["location"].to(
            DEVICE,
            non_blocking=True
        )

        targets = batch["label"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            sensor,
            video,
            location
        )

        loss = criterion(
            logits,
            targets
        )

        total_loss += (
            loss.item()
            * targets.size(0)
        )

        preds = logits.argmax(
            dim=1
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_targets.extend(
            targets.cpu().numpy()
        )

    avg_loss = (
        total_loss / len(loader.dataset)
    )

    macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        all_targets,
        all_preds
    )

    return (
        avg_loss,
        macro_f1,
        accuracy,
        np.array(all_targets),
        np.array(all_preds)
    )

In [ ]:
best_f1 = -np.inf
best_state = None

history = []

patience = 6
patience_counter = 0

for epoch in range(1, EPOCHS + 1):

    train_loss, train_f1, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_f1, val_acc, _, _ = evaluate(
        model,
        val_loader,
        criterion
    )

    scheduler.step()

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_f1": train_f1,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_f1": val_f1,
        "val_acc": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train F1 {train_f1:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val F1 {val_f1:.4f} | "
        f"Val Acc {val_acc:.4f}"
    )

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }

        patience_counter = 0

        print(
            f"  ✓ New best Macro-F1: "
            f"{best_f1:.4f}"
        )

    else:

        patience_counter += 1

        if patience_counter >= patience:

            print(
                "Early stopping."
            )

            break

Epoch 01 | Train Loss nan | Train F1 0.0307 | Val Loss nan | Val F1 0.0273 | Val Acc 0.3507
  ✓ New best Macro-F1: 0.0273


In [ ]:
model.load_state_dict(
    best_state
)

print(
    f"Best validation Macro-F1: "
    f"{best_f1:.4f}"
)

In [ ]:
val_loss, val_f1, val_acc, y_true, y_pred = evaluate(
    model,
    val_loader,
    criterion
)

print("Validation loss:", val_loss)
print("Validation accuracy:", val_acc)
print("Validation Macro-F1:", val_f1)

print("\nClassification report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=[
            ID_TO_LABEL[i]
            for i in range(NUM_CLASSES)
        ],
        zero_division=0
    )
)

In [ ]:
test_sensor = np.load(
    os.path.join(
        BASE_PATH,
        "test",
        "test_inertial_data.npy"
    )
)

test_video = np.load(
    os.path.join(
        BASE_PATH,
        "test",
        "test_videomae_data.npy"
    )
)

test_meta = pd.read_csv(
    os.path.join(
        BASE_PATH,
        "test",
        "test_meta_data.csv"
    )
)

print(test_sensor.shape)
print(test_video.shape)
print(test_meta.shape)

In [ ]:
class TestWearDataset(Dataset):

    def __init__(
        self,
        sensor,
        video,
        meta,
        sensor_mean,
        sensor_std
    ):

        self.sensor = sensor
        self.video = video
        self.meta = meta

        self.sensor_mean = sensor_mean
        self.sensor_std = sensor_std

    def __len__(self):

        return len(self.sensor)

    def __getitem__(self, idx):

        sensor = self.sensor[
            idx
        ].astype(np.float32)

        video = self.video[
            idx
        ].astype(np.float32)

        location = self.meta.iloc[
            idx
        ]["sensor_location"]

        location_id = LOCATION_TO_ID[
            location
        ]

        columns = SENSOR_MAPPINGS[
            location
        ]

        column_indices = [
            sensor_columns.index(c)
            for c in columns
        ]

        mean = self.sensor_mean[
            column_indices
        ]

        std = self.sensor_std[
            column_indices
        ]

        sensor = (
            sensor - mean
        ) / std

        return {
            "sensor": torch.tensor(
                sensor,
                dtype=torch.float32
            ),

            "video": torch.tensor(
                video,
                dtype=torch.float32
            ),

            "location": torch.tensor(
                location_id,
                dtype=torch.long
            )
        }

In [ ]:
test_dataset = TestWearDataset(
    test_sensor,
    test_video,
    test_meta,
    sensor_mean,
    sensor_std
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
@torch.no_grad()
def predict_test(
    model,
    loader
):

    model.eval()

    predictions = []

    for batch in loader:

        sensor = batch["sensor"].to(
            DEVICE,
            non_blocking=True
        )

        video = batch["video"].to(
            DEVICE,
            non_blocking=True
        )

        location = batch["location"].to(
            DEVICE,
            non_blocking=True
        )

        logits = model(
            sensor,
            video,
            location
        )

        preds = logits.argmax(
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

    return np.array(predictions)


test_predictions = predict_test(
    model,
    test_loader
)

print(
    "Predictions:",
    test_predictions.shape
)

print(
    "Unique predicted classes:",
    np.unique(test_predictions)
)

In [ ]:
submission = pd.DataFrame({
    "id": test_meta["id"],
    "target_value": test_predictions
})

submission = submission.sort_values(
    "id"
).reset_index(drop=True)

print(submission.head())

print(
    "Submission shape:",
    submission.shape
)

print(
    submission["target_value"]
    .value_counts()
)

In [ ]:
submission.to_csv(
    "/kaggle/working/submission_transformer.csv",
    index=False
)

print(
    "Saved:"
    "/kaggle/working/submission_transformer.csv"
)